In [63]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,ImagePath,Type
0,791,images/000791.png,Psychic
1,638,images/000638.png,Water
2,445,images/000445.png,Ghost
3,333,images/000333.png,Psychic
4,292,images/000292.png,Fighting


In [64]:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
df_train.Type = enc.fit_transform(df_train.Type)

df_train.head()

,SampleID,ImagePath,Type
0,791,images/000791.png,14
1,638,images/000638.png,17
2,445,images/000445.png,8
3,333,images/000333.png,14
4,292,images/000292.png,5


In [102]:
import torch

epochs = 20
learning_rate = 0.002
device = torch.device("cuda")

device

device(type='cuda')

In [121]:
from torch.utils.data import Dataset, DataLoader
import cv2
from torchvision import transforms

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
])

class MyDataset(Dataset):
    def __init__(self, df, transform=None, img_size=120):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.ImagePath
        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.img_size, self.img_size))
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255
        label = row.Type
        label = torch.tensor(label, dtype=torch.int64)
        return img, label

from sklearn.model_selection import train_test_split
X_train, X_valid = train_test_split(df_train, random_state=42, test_size=0.2)

train_dataset = MyDataset(X_train, transform=train_tf)
train_loader = DataLoader(train_dataset, batch_size=32)

test_dataset = MyDataset(X_valid)
test_loader = DataLoader(test_dataset, batch_size=32)

In [122]:
import torch.nn as nn # GANG NU MERGE NICIUN CNN NICI RESNET WTFFFFF
import torchvision

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 18)
        )
    def forward(self, X):
        return self.net(X)

model = Net().to(device)
#model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
#model.fc = nn.Linear(512, 18)
#model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [123]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Training Loss: {total_loss / len(train_loader):.4f}")


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader):.4f}')

In [124]:
for epoch in range(1, epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, test_loader, criterion)

Epoch 1, Training Loss: 2.7022
Test Loss: 2.6526
Epoch 2, Training Loss: 2.5837
Test Loss: 2.5339
Epoch 3, Training Loss: 2.5042
Test Loss: 2.5441
Epoch 4, Training Loss: 2.4790
Test Loss: 2.5835
Epoch 5, Training Loss: 2.4358
Test Loss: 2.4395
Epoch 6, Training Loss: 2.4085
Test Loss: 2.8495
Epoch 7, Training Loss: 2.4094
Test Loss: 2.8898
Epoch 8, Training Loss: 2.4014
Test Loss: 2.3648
Epoch 9, Training Loss: 2.3347
Test Loss: 2.4292
Epoch 10, Training Loss: 2.3220
Test Loss: 2.7465
Epoch 11, Training Loss: 2.3244
Test Loss: 2.3368
Epoch 12, Training Loss: 2.3170
Test Loss: 2.3720
Epoch 13, Training Loss: 2.2838
Test Loss: 2.6247
Epoch 14, Training Loss: 2.2526
Test Loss: 2.2400
Epoch 15, Training Loss: 2.2211
Test Loss: 2.4974
Epoch 16, Training Loss: 2.2018
Test Loss: 2.3179
Epoch 17, Training Loss: 2.1749
Test Loss: 2.3497
Epoch 18, Training Loss: 2.1680
Test Loss: 2.1305
Epoch 19, Training Loss: 2.1292
Test Loss: 2.3516
Epoch 20, Training Loss: 2.1151
Test Loss: 2.3751


In [125]:
class FinalDataset(Dataset):
    def __init__(self, df, img_size=120):
        self.df = df
        self.img_size = img_size
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.ImagePath
        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255
        return img

final_dataset = FinalDataset(df_test)
final_loader = DataLoader(final_dataset, batch_size=32)

In [126]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predicted = torch.argmax(outputs, dim=1)
        preds.extend(predicted.cpu().numpy())

preds[:10]

[12, 12, 12, 17, 17, 17, 6, 17, 17, 13]

In [127]:
preds = enc.inverse_transform(preds)
preds[:10]

array(['Normal', 'Normal', 'Normal', 'Water', 'Water', 'Water', 'Fire',
       'Water', 'Water', 'Poison'], dtype=object)

In [128]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "Type": preds
})

df_preds

,SampleID,Type
0,699,Normal
1,297,Normal
2,228,Normal
3,337,Water
4,538,Water
...,...,...
157,669,Normal
158,674,Normal
159,756,Grass
160,754,Grass


In [129]:
df_preds.to_csv("submission.csv", index=False)